# Create train, validation, and test splits

This notebook reads the combined `Data/Raw/Descriptor_fs.csv`, then writes split metadata into `Data/Preprocessed`.

Test contains every annotation from 20180812, 20180915, and 20220910, exclusively. The remaining dates are stratified by class into training and validation with relative weights 70:15 (validation fraction 15/85). All labels, including snow and anthropogenic surfaces, are retained.

It does not copy or move matrix `.txt` files. It validates annotated target-cell files by direct path lookup under `Data/Raw/beach_all/<date>/`; later notebooks use the saved `date` and `cell_id` values to load target cells and their same-date context from that directory.

In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

DESCRIPTOR_PATH = PROJECT_ROOT / "Data" / "Raw" / "Descriptor_fs.csv"
CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
FEATURE_SUFFIXES = ("r", "g", "b", "s", "d")

# Relative weights for the non-test remainder; test size is date-defined.
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_DATES = {"20180812", "20180915", "20220910"}
RANDOM_SEED = 42

validation_relative_fraction = VALIDATION_FRACTION / (TRAIN_FRACTION + VALIDATION_FRACTION)

print(f"Project root:     {PROJECT_ROOT}")
print(f"Descriptor:       {DESCRIPTOR_PATH}")
print(f"Cell data root:   {CELL_DATA_ROOT}")
print(f"Output directory: {PREPROCESSED_DIR}")

In [ ]:
DESCRIPTOR_COLUMNS = ["date", "cell_id", "class"]
DESCRIPTOR_DTYPES = {column: "string" for column in DESCRIPTOR_COLUMNS}


def read_descriptor(path):
    if not path.exists():
        raise FileNotFoundError(f"Descriptor file not found: {path}")
    frame = pd.read_csv(
        path,
        sep=";",
        header=None,
        names=DESCRIPTOR_COLUMNS,
        dtype=DESCRIPTOR_DTYPES,
        encoding="utf-8-sig",
    )

    frame = frame.dropna(subset=DESCRIPTOR_COLUMNS).copy()
    for column in DESCRIPTOR_COLUMNS:
        frame[column] = frame[column].str.strip()
    empty_mask = frame[DESCRIPTOR_COLUMNS].eq("").any(axis=1)
    if empty_mask.any():
        raise ValueError(f"Empty descriptor values found in {path}: {int(empty_mask.sum())} rows")
    return frame


descriptor_raw = read_descriptor(DESCRIPTOR_PATH)

duplicate_mask = descriptor_raw.duplicated(subset=["date", "cell_id"], keep=False)
if duplicate_mask.any():
    duplicates = descriptor_raw.loc[duplicate_mask].sort_values(["date", "cell_id", "class"])
    conflicting_keys = (
        duplicates.groupby(["date", "cell_id"])["class"].nunique().loc[lambda counts: counts > 1]
    )
    if not conflicting_keys.empty:
        conflicts = duplicates.merge(
            conflicting_keys.rename("class_count"),
            on=["date", "cell_id"],
        )
        display(conflicts)
        raise ValueError("Conflicting classes found for duplicate date/cell_id pairs.")
    duplicate_key_count = duplicates[["date", "cell_id"]].drop_duplicates().shape[0]
    print(f"Dropping {len(duplicates) - duplicate_key_count:,} repeated annotations.")

descriptor = (
    descriptor_raw
    .drop_duplicates(subset=["date", "cell_id"], keep="first")
    .sort_values(["date", "cell_id"])
    .reset_index(drop=True)
)
descriptor.head(), descriptor.shape

In [ ]:
if not CELL_DATA_ROOT.exists():
    raise FileNotFoundError(f"Cell data root not found: {CELL_DATA_ROOT}")

missing_rows = []
for row in descriptor.itertuples(index=False):
    missing_features = [
        suffix
        for suffix in FEATURE_SUFFIXES
        if not (CELL_DATA_ROOT / row.date / f"{row.cell_id}{suffix}.txt").is_file()
    ]
    if missing_features:
        missing_rows.append(
            {
                "date": row.date,
                "cell_id": row.cell_id,
                "class": row[2],
                "missing_features": ",".join(missing_features),
            }
        )

missing_cells = pd.DataFrame(missing_rows)
if not missing_cells.empty:
    display(missing_cells.head(50))
    raise FileNotFoundError(
        f"{len(missing_cells):,} annotated cells are incomplete under {CELL_DATA_ROOT}."
    )

print(f"Validated five feature files for all {len(descriptor):,} annotated cells in {CELL_DATA_ROOT}.")

class_counts = descriptor["class"].value_counts().sort_index()
class_counts.to_frame(name="count")

In [ ]:
test_mask = descriptor["date"].isin(TEST_DATES)
test_df = descriptor.loc[test_mask].copy()
remaining_df = descriptor.loc[~test_mask].copy()
if set(test_df["date"]) != TEST_DATES:
    raise ValueError("Descriptor must contain annotations for all three BS test dates.")

# Do not filter classes: snow and anthropogenic labels remain intact.
train_df, validation_df = train_test_split(
    remaining_df,
    test_size=validation_relative_fraction,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=remaining_df["class"],
)

train_df = train_df.assign(split="train")
validation_df = validation_df.assign(split="validation")
test_df = test_df.assign(split="test")

split_df = pd.concat([train_df, validation_df, test_df], ignore_index=True)
split_df = split_df.sort_values(["split", "date", "cell_id"]).reset_index(drop=True)

split_counts = split_df["split"].value_counts().reindex(["train", "validation", "test"])
split_counts.to_frame(name="count")

In [ ]:
expected_keys = set(map(tuple, descriptor[["date", "cell_id"]].to_numpy()))
actual_keys = set(map(tuple, split_df[["date", "cell_id"]].to_numpy()))

if expected_keys != actual_keys:
    missing = expected_keys - actual_keys
    unexpected = actual_keys - expected_keys
    raise AssertionError(f"Split mismatch. Missing={len(missing)}, unexpected={len(unexpected)}")

if split_df.duplicated(subset=["date", "cell_id"]).any():
    raise AssertionError("A date/cell_id pair appears in more than one split.")

distribution = (
    split_df
    .groupby(["split", "class"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "validation", "test"])
)

display(distribution)
assert set(test_df["date"]) == TEST_DATES
assert not train_df["date"].isin(TEST_DATES).any()
assert not validation_df["date"].isin(TEST_DATES).any()
assert len(test_df) == int(descriptor["date"].isin(TEST_DATES).sum())
assert set(split_df["class"]) == set(descriptor["class"])


In [ ]:
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

columns = ["date", "cell_id", "class", "split"]
output_files = {
    "all": PREPROCESSED_DIR / "bs_dataset_splits.csv",
    "train": PREPROCESSED_DIR / "bs_train_split.csv",
    "validation": PREPROCESSED_DIR / "bs_validation_split.csv",
    "test": PREPROCESSED_DIR / "bs_test_split.csv",
}

split_df[columns].to_csv(output_files["all"], index=False)
for split_name in ["train", "validation", "test"]:
    split_df.loc[split_df["split"] == split_name, columns].to_csv(output_files[split_name], index=False)

for split_name, path in output_files.items():
    print(f"{split_name:>10}: {path}")
split_provenance = {
    "policy": "bs_date_holdout_20180812_20180915_20220910_v1",
    "test_dates": sorted(TEST_DATES),
    "validation_fraction_of_remainder": validation_relative_fraction,
    "random_seed": RANDOM_SEED,
    "csv_sha256": {
        name: hashlib.sha256(output_files[name].read_bytes()).hexdigest()
        for name in ("train", "validation", "test")
    },
}
with (PREPROCESSED_DIR / "bs_split_provenance.json").open("w", encoding="utf-8") as file:
    json.dump(split_provenance, file, indent=2)
display(pd.DataFrame([split_provenance]))
